# ComfyUI de Leo — Z-Image Turbo (KAGGLE)

Versión Kaggle del notebook (alternativa free a Colab, 30 h/semana de GPU).

### ⚠️ ANTES de correr (una sola vez):
1. Cuenta Kaggle con **teléfono verificado** (necesario para Internet + GPU). *Settings → Phone Verification.*
2. En este notebook, panel derecho:
   - **Accelerator → GPU T4 x2** (o P100).
   - **Internet → ON**.
3. Arriba: *Run → Run all*.

La 1ª vez baja los modelos (~14 GB, pero la red de Kaggle es rápida: pocos minutos). Al final imprime una **URL pública** → pegala en comfyweb.

> Kaggle **no monta Google Drive**: por defecto los modelos se bajan cada sesión. Para no rebajarlos, ver al final *Persistencia (Kaggle Dataset)*.

## 1) Clonar ComfyUI + custom nodes

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.isdir('/kaggle/working/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI
os.chdir('/kaggle/working/ComfyUI')
!pip install -q -r requirements.txt

CN = '/kaggle/working/ComfyUI/custom_nodes'
repos = [
    'https://github.com/ltdrdata/ComfyUI-Manager',
    'https://github.com/Fannovel16/comfyui_controlnet_aux',
    'https://github.com/TTPlanetPig/Comfyui_TTP_Toolset',
    'https://github.com/yolain/ComfyUI-Easy-Use',
    'https://github.com/rgthree/rgthree-comfy',
    'https://github.com/LAOGOU-666/Comfyui-Memory_Cleanup',
]
for u in repos:
    name = u.rstrip('/').split('/')[-1]
    p = os.path.join(CN, name)
    if not os.path.isdir(p):
        !git clone --depth 1 {u} {p}
    req = os.path.join(p, 'requirements.txt')
    if os.path.isfile(req):
        !pip install -q -r {req}
print('\nComfyUI + custom nodes listos')

## 2) Descargar modelos (fuentes oficiales Apache-2.0)

Van a `/kaggle/temp/models` (rápido). Si adjuntaste un Dataset con los modelos, saltea esta celda.

In [ ]:
import os
MODELS = '/kaggle/temp/models'
for sub in ['diffusion_models','text_encoders','vae','model_patches','upscale_models','loras']:
    os.makedirs(os.path.join(MODELS, sub), exist_ok=True)

def dl(url, dest, min_mb=1):
    if os.path.isfile(dest) and os.path.getsize(dest) > min_mb*1_000_000:
        print('✓ ya está:', os.path.basename(dest)); return
    print('↓ bajando:', os.path.basename(dest))
    !wget -q --show-progress -O "{dest}" "{url}"

dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', f'{MODELS}/diffusion_models/z-image-turbo-fp8-e4m3fn.safetensors', 500)
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors', f'{MODELS}/text_encoders/qwen_3_4b_fp8_mixed.safetensors', 500)
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', f'{MODELS}/vae/ae.safetensors', 50)
dl('https://huggingface.co/alibaba-pai/Z-Image-Turbo-Fun-Controlnet-Union/resolve/main/Z-Image-Turbo-Fun-Controlnet-Union.safetensors', f'{MODELS}/model_patches/Z-Image-Turbo-Fun-Controlnet-Union.safetensors', 500)
dl('https://huggingface.co/Kim2091/2x-AnimeSharpV4/resolve/main/2x-AnimeSharpV4_RCAN.safetensors', f'{MODELS}/upscale_models/2x-AnimeSharpV4_RCAN.safetensors', 5)
print('\nModelos OK en', MODELS)

## 3) Apuntar ComfyUI a los modelos + lanzar + túnel público

In [ ]:
import os, threading, time, socket
os.chdir('/kaggle/working/ComfyUI')

# Apuntar ComfyUI a /kaggle/temp/models (o al path de tu Dataset)
MODELS = '/kaggle/temp/models'
yaml = f'''leo_kaggle:
  base_path: {MODELS}
  diffusion_models: diffusion_models
  text_encoders: text_encoders
  clip: text_encoders
  vae: vae
  model_patches: model_patches
  upscale_models: upscale_models
  loras: loras
'''
open('/kaggle/working/ComfyUI/extra_model_paths.yaml','w').write(yaml)
print('extra_model_paths.yaml escrito')

!pip install -q pycloudflared

def tunnel():
    while socket.socket().connect_ex(('127.0.0.1', 8188)) != 0:
        time.sleep(1)
    from pycloudflared import try_cloudflare
    url = try_cloudflare(port=8188).tunnel
    print('\n\n' + '='*48)
    print('  URL PUBLICA (pegala en comfyweb):')
    print('  ' + url)
    print('='*48 + '\n')

threading.Thread(target=tunnel, daemon=True).start()
!python main.py --dont-print-server

---
### (Opcional) Persistencia — no rebajar 14 GB cada vez

Kaggle no tiene Drive, pero podés guardar los modelos como **Kaggle Dataset** (una vez) y adjuntarlo:

1. Corré las celdas 1 y 2 una vez (baja los modelos a `/kaggle/temp/models`).
2. Copialos a `/kaggle/working/models` y hacé *Save Version* (commit) — el output queda guardado.
3. *Create Dataset* desde ese output (o desde `/kaggle/working/models`).
4. En futuras sesiones: panel derecho → *Add Input* → tu dataset. Queda montado en `/kaggle/input/<tu-dataset>/` (solo lectura, instantáneo).
5. En la celda 3 cambiá `MODELS = '/kaggle/input/<tu-dataset>'` y salteá la celda 2.

Así arranca en segundos sin rebajar nada.